# Class 6 - Functions, Lambdas & Modules
### Data Analytics in Python | Week 2, Sunday

Functions are how you stop writing the same code twice.
Every time you copy a block of code and paste it somewhere else,
that block should be a function.

But functions are more than just code reuse. They're the unit of
**thinking** in a program. A well-named function is documentation.
A function with a single responsibility is testable. A function with
no side effects is predictable. These properties matter enormously
when your analytics pipeline runs on a million rows.

**Today:**
- `def`, parameters, return values
- Default parameters - and a famous trap
- `*args` and `**kwargs` for flexible signatures
- Variable scope: local vs global (LEGB rule)
- `lambda` - anonymous single-expression functions
- `map()`, `filter()`, `sorted()` with `key=`
- Built-ins: `any()`, `all()`
- Modules: `math`, `random`, `datetime`, `collections`
- Practical: Reusable analytics toolkit

- Parameters are the placeholders defined in the function's declaration (the "blueprint").
- Arguments are the actual values you pass into the function when you call it.

## 1. Function basics

Define once, call many times. A function is a named, reusable block of code that accepts inputs, does work, and optionally returns a result.

In [ ]:
# Basic definition
def greet(name):
    """Return a greeting string for the given name."""
    return f"Hello, {name}!"

# Call it
print(greet("Ahmad"))
print(greet("Sara"))
print(greet("the class"))

# Functions without return return None implicitly
def log_message(msg):
    print(f"[LOG] {msg}")
    # no return statement

result = log_message("Pipeline started")
print(f"\nReturn value of log_message: {result!r}")   # None

Hello, Ahmad!
Hello, Sara!
Hello, the class!
[LOG] Pipeline started

Return value of log_message: None


### 1a. Multiple parameters, multiple return values
- Call with positional args
- Call with keyword args - order doesn't matter
- you can also capture as a tuple with single var - if multiple return values

In [ ]:
def bmi_report(weight_kg, height_m, name="Patient"):
    """Compute BMI and return category."""
    bmi = weight_kg / (height_m ** 2)

    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25.0:
        category = "Normal"
    elif bmi < 30.0:
        category = "Overweight"
    else:
        category = "Obese"

    return bmi, category   # returns a tuple - unpack at call site

# Call with positional args
bmi, cat = bmi_report(75, 1.75)
print(f"BMI: {bmi:.1f}  Category: {cat}")

# Call with keyword args - order doesn't matter
bmi, cat = bmi_report(height_m=1.65, weight_kg=55, name="Sara")
print(f"BMI: {bmi:.1f}  Category: {cat}")

#you can also capture as a tuple
result = bmi_report(90, 1.70)
print(f"\nFull result tuple: {result}")

BMI: 24.5  Category: Normal
BMI: 20.2  Category: Normal

Full result tuple: (31.14186851211073, 'Obese')


## 2. Default parameters - and a famous trap

Default values make parameters optional. But there's one rule you must memorise: **never use a mutable object (list, dict, set) as a default value**.

In [3]:
# Default parameters - simple types are fine
def describe_patient(name, age=None, unit="Celsius"):
    age_str = f", age {age}" if age is not None else ""
    return f"Patient: {name}{age_str}. Temperature unit: {unit}"

print(describe_patient("Ahmad"))
print(describe_patient("Sara", age=28))
print(describe_patient("Bilal", age=45, unit="Fahrenheit"))

Patient: Ahmad. Temperature unit: Celsius
Patient: Sara, age 28. Temperature unit: Celsius
Patient: Bilal, age 45. Temperature unit: Fahrenheit


### 2a. Never use a Mutable object as default Value

In [ ]:
# THE MUTABLE DEFAULT TRAP - one of Python's most famous gotchas

def append_reading(value, history=[]):
    """Add a temperature reading to a patient's history."""
    history.append(value)
    return history

#you'd expect each patient to start with an empty list
p1_history = append_reading(38.1)
p2_history = append_reading(37.4)
p3_history = append_reading(39.0)

print("Patient histories:")
print(f"  P1: {p1_history}")   # expected [38.1] - but...
print(f"  P2: {p2_history}")   # expected [37.4] - but...
print(f"  P3: {p3_history}")   # expected [39.0] - but...

#all three share the SAME list object!
print(f"\nAll point to same list? {p1_history is p2_history}")

Patient histories:
  P1: [38.1, 37.4, 39.0]
  P2: [38.1, 37.4, 39.0]
  P3: [38.1, 37.4, 39.0]

All point to same list? True


#### Solution to Mutable Trap

In [ ]:
#use None as default, create fresh object inside the function

def append_reading_safe(value, history=None):
    """Safe version - creates fresh list if none provided."""
    if history is None:
        history = []       # new list each time
    history.append(value)
    return history

p1 = append_reading_safe(38.1)
p2 = append_reading_safe(37.4)
p3 = append_reading_safe(39.0)

print("Correct patient histories:")
print(f"  P1: {p1}")
print(f"  P2: {p2}")
print(f"  P3: {p3}")
print(f"\nAll separate? {p1 is not p2 is not p3}")

# Real-world consequence: this same bug kills pandas pipelines
# df_modified = transform(df)  <--- if transform has a mutable default i.e. list or dict etc, silent corruption as it will mix data or append it with other columns

Correct patient histories:
  P1: [38.1]
  P2: [37.4]
  P3: [39.0]

All separate? True


## **3. \*args and \*\*kwargs**

Two mechanisms for writing functions that accept a variable number of arguments.

### 3a. `*args` - collects extra positional arguments into a TUPLE

In [6]:
# *args - collects extra positional arguments into a TUPLE
def mean(*values):
    """Compute mean of any number of values."""
    if not values:
        return None
    return sum(values) / len(values)

print(f"mean(1,2,3):     {mean(1, 2, 3)}")
print(f"mean(10,20):     {mean(10, 20)}")
print(f"mean(5):         {mean(5)}")
print(f"mean():          {mean()}")

# *args is just a tuple inside the function
def inspect_args(*args):
    print(f"args type: {type(args).__name__}")
    print(f"args: {args}")
    for i, a in enumerate(args):
        print(f"  args[{i}] = {a!r}")

inspect_args("data", 42, [1,2,3])

mean(1,2,3):     2.0
mean(10,20):     15.0
mean(5):         5.0
mean():          None
args type: tuple
args: ('data', 42, [1, 2, 3])
  args[0] = 'data'
  args[1] = 42
  args[2] = [1, 2, 3]


### 3b. `**kwargs` - collects extra keyword arguments into a DICT

- Positional Argument - Must be passed either by its position (first) or explicitly by name
- Variable-length Positional Arguments - The single asterisk (*) collects any extra positional `arguments` passed to the function (packs these extra values into a tuple) that weren't captured by `dataset_name`.
- Keyword-Only / Named Argument - Must be passed using key=value syntax. It also has a default value (True)
- Variable-length Keyword Arguments - The double asterisk () collects any extra `keyword arguments` that weren't explicitly defined in the function signature.


#### Quick Comparison Table

| Argument Type | Syntax in `def` | Packed As | Required? | How to call it |
| --- | --- | --- | --- | --- |
| **Positional** | `dataset_name` | None (Single Variable) | Yes | By position or name |
| **Variable-length Positional** | `*transforms` | `tuple`<br> | No (can be empty) | By position only |
| **Keyword-Only / Named Argument** | `verbose=True` | None (Single Variable) | No (has default) | **Must** use name (`verbose=False`)|
| **Variable-length Keyword** | `**metadata` | `dict`<br> | No (can be empty) | Any name-value pair (`key=value`) |


In [1]:
# **kwargs - collects extra keyword arguments into a DICT
def create_patient(**kwargs):
    """Create a patient record from keyword arguments."""
    print(f"Creating patient with {len(kwargs)} fields:")
    for key, val in kwargs.items():
        print(f"  {key}: {val}")
    return kwargs

p = create_patient(id="P0042", name="Ahmad", age=34, temp=38.7)
print(f"\nPatient dict: {p}")

Creating patient with 4 fields:
  id: P0042
  name: Ahmad
  age: 34
  temp: 38.7

Patient dict: {'id': 'P0042', 'name': 'Ahmad', 'age': 34, 'temp': 38.7}


### 3c. Order of using All Arguments

In [2]:
#combine both: positional, *args, keyword, **kwargs
def pipeline(dataset_name, *transforms, verbose=True, **metadata):
    print(f"\nRunning pipeline on: {dataset_name}")
    print(f"Transforms: {transforms}")
    print(f"Verbose: {verbose}")
    print(f"Metadata: {metadata}")

pipeline("patient_records", "clean", "normalise", "encode",
         verbose=True, version="1.2", author="Lukman")


Running pipeline on: patient_records
Transforms: ('clean', 'normalise', 'encode')
Verbose: True
Metadata: {'version': '1.2', 'author': 'Lukman'}


## 4. Variable scope - LEGB rule

When Python looks up a variable name, it searches in this order:
**L**ocal ---> **E**nclosing ---> **G**lobal ---> **B**uilt-in

In [9]:
x = "global"    # global scope

def outer():
    x = "enclosing"   # enclosing scope

    def inner():
        x = "local"   # local scope
        print(f"inner sees:    {x}")

    inner()
    print(f"outer sees:    {x}")

outer()
print(f"module sees:   {x}")

inner sees:    local
outer sees:    enclosing
module sees:   global


### 4a. Functions see global variables but create LOCAL copies on assignment

In [3]:
# The key rule: functions see global variables but create LOCAL copies on assignment
count = 0    # global

def increment_broken():
    count += 1   # this FAILS - Python sees 'count' on left side = local variable
                 # but it's never assigned before this line

try:
    increment_broken()
except UnboundLocalError as e:
    print(f"UnboundLocalError: {e}")

# solution 1: pass as argument and return new value (PREFERRED)
def increment(n):
    return n + 1

count = increment(count)
print(f"count after increment: {count}")

# solution 2: global keyword (use sparingly - makes code hard to reason about)
def increment_global():
    global count
    count += 1

increment_global()
print(f"count after global increment: {count}")

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value
count after increment: 1
count after global increment: 2


## 5. Lambda functions

A `lambda` is an anonymous, single-expression function. Use it for short operations passed as arguments to other functions. If it needs more than one line, use `def`.

In [1]:
# Basic lambda syntax: lambda parameters: expression
square   = lambda x: x ** 2
add      = lambda x, y: x + y
classify = lambda score: "pass" if score >= 60 else "fail"

print(f"square(5):       {square(5)}")
print(f"add(3, 4):       {add(3, 4)}")
print(f"classify(75):    {classify(75)}")
print(f"classify(45):    {classify(45)}")

# Lambdas are most useful as inline key functions
genes = ["BRCA1", "TP53", "KRAS", "EGFR", "MYC"]

# Sort by last character
by_last = sorted(genes, key=lambda g: g[-1])
print(f"\nBy last char:   {by_last}")

# Sort by length, then alphabetically (stable sort)
by_len_alpha = sorted(genes, key=lambda g: (len(g), g))
print(f"By len+alpha:   {by_len_alpha}")

square(5):       25
add(3, 4):       7
classify(75):    pass
classify(45):    fail

By last char:   ['BRCA1', 'TP53', 'MYC', 'EGFR', 'KRAS']
By len+alpha:   ['MYC', 'EGFR', 'KRAS', 'TP53', 'BRCA1']


In [ ]:
# lambda with sorted() for complex analytics sorting
patients = [
    {"name": "Ahmad",  "age": 34, "risk_score": 7},
    {"name": "Sara",   "age": 28, "risk_score": 2},
    {"name": "Bilal",  "age": 51, "risk_score": 9},
    {"name": "Zara",   "age": 45, "risk_score": 7},
    {"name": "Hassan", "age": 67, "risk_score": 4},
]

# Sort by risk score descending, then age descending (if risk is same)
triaged = sorted(patients,
                 key=lambda p: (-p["risk_score"], -p["age"]))

print(f"{'Name':<10} {'Risk':>6} {'Age':>5}")
print("-" * 24)
for p in triaged:
    print(f"{p['name']:<10} {p['risk_score']:>6}  {p['age']:>5}")

Name         Risk   Age
------------------------
Bilal           9     51
Zara            7     45
Ahmad           7     34
Hassan          4     67
Sara            2     28


## 6. map(), filter(), sorted() with key=

Functional programming tools that take a function and an iterable. They return **iterators** - wrap with `list()` to see the result.

### 6a. `map(function, iterable)` - apply function to every element
### `filter(function, iterable)` - keep elements where function returns True

In [3]:
scores = [88, 55, 94, 43, 77, 91, 60, 82, 38, 96]

# map(function, iterable) - apply function to every element
# Returns an iterator, not a list
doubled   = list(map(lambda x: x * 2, scores))
normalised = list(map(lambda x: round((x - min(scores)) /
                                      (max(scores) - min(scores)) * 100, 1),
                      scores))

print(f"Original:   {scores}")
print(f"Doubled:    {doubled}")
print(f"Normalised: {normalised}")

# filter(function, iterable) - keep elements where function returns True
passing  = list(filter(lambda x: x >= 60, scores))
at_risk  = list(filter(lambda x: 40 <= x < 60, scores))
failing  = list(filter(lambda x: x < 40, scores))

print(f"\nPassing (>=60): {passing}")
print(f"At risk (40-59):{at_risk}")
print(f"Failing (<40):  {failing}")

Original:   [88, 55, 94, 43, 77, 91, 60, 82, 38, 96]
Doubled:    [176, 110, 188, 86, 154, 182, 120, 164, 76, 192]
Normalised: [86.2, 29.3, 96.6, 8.6, 67.2, 91.4, 37.9, 75.9, 0.0, 100.0]

Passing (>=60): [88, 94, 77, 91, 60, 82, 96]
At risk (40-59):[55, 43]
Failing (<40):  [38]


### 6b. When to use map/filter vs comprehensions

In [ ]:
# Both achieve the same thing - comprehensions are usually more readable
data = [88, 55, 94, 43, 77]

# These are equivalent:
via_filter = list(filter(lambda x: x >= 60, data))
via_comp   = [x for x in data if x >= 60]
print(f"filter: {via_filter}")
print(f"comp:   {via_comp}")

# map + filter combined - the functional chain
via_map_filter = list(map(lambda x: round(x/100, 2),
                          filter(lambda x: x >= 60, data)))
via_comp_both  = [round(x/100, 2) for x in data if x >= 60]
print(f"\nMap+filter: {via_map_filter}")
print(f"Comp:       {via_comp_both}")

# any() and all() - boolean aggregations
print(f"\nany score >= 90:  {any(x >= 90 for x in data)}")
print(f"all scores >= 40: {all(x >= 40 for x in data)}")
print(f"all scores >= 60: {all(x >= 60 for x in data)}")

filter: [88, 94, 77]
comp:   [88, 94, 77]

Map+filter: [0.88, 0.94, 0.77]
Comp:       [0.88, 0.94, 0.77]

any score >= 90:  True
all scores >= 40: True
all scores >= 60: False


## 7. Modules

Modules are Python files (or C extensions) you import to access extra functionality. The standard library has over 200 modules - you don't need external packages for most analytics work.

### 7a. Math Module

In [4]:
import math

print("=== math module ===")
print(f"math.pi:          {math.pi:.6f}")
print(f"math.e:           {math.e:.6f}")
print(f"math.sqrt(2):     {math.sqrt(2):.6f}")
print(f"math.log(100):    {math.log(100):.4f}")     # natural log
print(f"math.log(100,10): {math.log(100, 10):.4f}") # log base 10
print(f"math.ceil(3.2):   {math.ceil(3.2)}")
print(f"math.floor(3.8):  {math.floor(3.8)}")
print(f"math.factorial(6):{math.factorial(6)}")
print(f"math.gcd(48,18):  {math.gcd(48, 18)}")      #greatest common diviser

# Practical: compute standard deviation without numpy
data = [88, 55, 94, 43, 77, 91, 60]
mean = sum(data) / len(data)
variance = sum((x - mean) ** 2 for x in data) / len(data)
std_dev  = math.sqrt(variance)
print(f"\nManual std dev of {data}: {std_dev:.2f}")

=== math module ===
math.pi:          3.141593
math.e:           2.718282
math.sqrt(2):     1.414214
math.log(100):    4.6052
math.log(100,10): 2.0000
math.ceil(3.2):   4
math.floor(3.8):  3
math.factorial(6):720
math.gcd(48,18):  6

Manual std dev of [88, 55, 94, 43, 77, 91, 60]: 18.51


### 7b. Random Module

In [22]:
import random
random.seed(42)   # ALWAYS seed for reproducibility in analytics

print("=== random module ===")
print(f"randint(1,100): {random.randint(1, 100)}")      # inclusive both ends
print(f"random():       {random.random():.4f}")          # [0, 1)
print(f"uniform(0,5):   {random.uniform(0, 5):.4f}")
print(f"gauss(70,10):   {random.gauss(70, 10):.2f}")    # normal distribution

# choice and choices
gene_panel = ["BRCA1", "TP53", "KRAS", "EGFR", "MYC"]
print(f"choice:   {random.choice(gene_panel)}")
print(f"choices 3:{random.choices(gene_panel, k=3)}")   # with replacement

# shuffle - in-place
scores = [88, 55, 94, 43, 77]
random.shuffle(scores)
print(f"shuffled: {scores}")

# sample - without replacement
print(f"sample 3: {random.sample(gene_panel, k=3)}")   # no repeats

=== random module ===
randint(1,100): 82
random():       0.1113
uniform(0,5):   3.7078
gauss(70,10):   70.18
choice:   BRCA1
choices 3:['EGFR', 'MYC', 'BRCA1']
shuffled: [55, 94, 77, 88, 43]
sample 3: ['TP53', 'MYC', 'KRAS']


### 7c. Datetime Module

`strptime` acts as a translator (reading an unknown text and understanding its dates), whereas `strftime` acts as a tailor (dressing a rigid, internal date into a custom outfit). `strptime` converts a string into a datetime object (String Parse Time), while `strftime` converts a datetime object into a formatted string (String Format Time)

1. `datetime.strptime(date_string, format_pattern)`: String to Datetime - Translates the string into a `datetime object` **(parsing)**.
2. `datetime.strftime()`: Datetime to String, it takes a `datetime object` and a `format string` **(formatting)**.

In [ ]:
from datetime import datetime, date, timedelta

print("=== datetime module ===")
now   = datetime.now()
today = date.today()

print(f"now:        {now}")
print(f"today:      {today}")
print(f"year:       {today.year}")
print(f"month:      {today.month}")
print(f"day:        {today.day}")
print(f"weekday():  {today.weekday()}  (0=Mon, 6=Sun)")

# Parsing and formatting

#strftime stands for "string format time, strptime - String Parse Time
admit_date = datetime.strptime("2024-03-15 09:30:00", "%Y-%m-%d %H:%M:%S")
print(f"\nParsed: {admit_date}")
print(f"Formatted: {admit_date.strftime('%d %B %Y, %I:%M %p')}")

# Arithmetic
discharge = admit_date + timedelta(days=5, hours=6)
stay = discharge - admit_date
print(f"Discharge: {discharge.strftime('%Y-%m-%d')}")
print(f"Stay: {stay.days} days, {stay.seconds//3600} hours")

=== datetime module ===
now:        2026-07-13 19:33:53.905867
today:      2026-07-13
year:       2026
month:      7
day:        13
weekday():  0  (0=Mon, 6=Sun)

Parsed: 2024-03-15 09:30:00
Formatted: 15 March 2024, 09:30 AM
Discharge: 2024-03-20
Stay: 5 days, 6 hours


### 7d. Collections module

In [18]:
from collections import Counter, defaultdict, OrderedDict

print("=== collections module ===")

# Counter - fast frequency counting (what you built manually with dicts)
words = ["data","analytics","python","data","machine","learning","data","python"]
counts = Counter(words)
print(f"Counter:         {counts}")
print(f"Most common 3:   {counts.most_common(3)}")
print(f"Count of 'data': {counts['data']}")

# defaultdict - dict that auto-creates missing keys
gene_samples = defaultdict(list)
readings = [("BRCA1",1.2),("TP53",0.8),("BRCA1",2.1),("TP53",1.5),("KRAS",0.9)]
for gene, expr in readings:
    gene_samples[gene].append(expr)  # no KeyError - list auto-created

print(f"\ndefaultdict: {dict(gene_samples)}")

# Counter arithmetic
study_A = Counter({"BRCA1":5, "TP53":3, "KRAS":8})
study_B = Counter({"BRCA1":2, "PIK3CA":4, "KRAS":3})
print(f"\nA + B: {study_A + study_B}")   # combine
print(f"A - B: {study_A - study_B}")      # subtract (removes 0 or negatives)

=== collections module ===
Counter:         Counter({'data': 3, 'python': 2, 'analytics': 1, 'machine': 1, 'learning': 1})
Most common 3:   [('data', 3), ('python', 2), ('analytics', 1)]
Count of 'data': 3

defaultdict: {'BRCA1': [1.2, 2.1], 'TP53': [0.8, 1.5], 'KRAS': [0.9]}

A + B: Counter({'KRAS': 11, 'BRCA1': 7, 'PIK3CA': 4, 'TP53': 3})
A - B: Counter({'KRAS': 5, 'BRCA1': 3, 'TP53': 3})


## Practical

### Practical 1 - The mutable default argument (live discovery)
**Context:** A student writes a function to track a patient's temperature history across clinic visits. It works fine at first. But after several patients, something strange happens.

In [19]:
# Students predict: does each patient get their own empty list?

def record_temp(patient_id, temp, history=[]):   # DANGEROUS DEFAULT
    history.append({"patient": patient_id, "temp": temp})
    return history

# Simulate three different patients visiting the clinic
p1_history = record_temp("P001", 38.9)
p2_history = record_temp("P002", 36.7)
p3_history = record_temp("P003", 39.1)

print("Patient histories (expected: 1 record each):")
print(f"P001: {p1_history}")
print(f"P002: {p2_history}")
print(f"P003: {p3_history}")
print(f"\nAll the same list? {p1_history is p2_history}")

# Why? The default list is created ONCE when the function is DEFINED,
# not each time it's CALLED. It lives in the function object:
print(f"\nThe shared list lives here: {record_temp.__defaults__}")

Patient histories (expected: 1 record each):
P001: [{'patient': 'P001', 'temp': 38.9}, {'patient': 'P002', 'temp': 36.7}, {'patient': 'P003', 'temp': 39.1}]
P002: [{'patient': 'P001', 'temp': 38.9}, {'patient': 'P002', 'temp': 36.7}, {'patient': 'P003', 'temp': 39.1}]
P003: [{'patient': 'P001', 'temp': 38.9}, {'patient': 'P002', 'temp': 36.7}, {'patient': 'P003', 'temp': 39.1}]

All the same list? True

The shared list lives here: ([{'patient': 'P001', 'temp': 38.9}, {'patient': 'P002', 'temp': 36.7}, {'patient': 'P003', 'temp': 39.1}],)


### Practical 2 - why doesn't the function modify the outer variable?
**Context:** A student writes a function to add a sample count, expects the outer variable to update. It doesn't. Why?

In [20]:
sample_count = 0

def add_samples(n):
    sample_count = sample_count + n   # local assignment shadows global
    print(f"  Inside function: sample_count = {sample_count}")

print(f"Before: sample_count = {sample_count}")
try:
    add_samples(5)
except UnboundLocalError as e:
    print(f"  UnboundLocalError: {e}")
print(f"After:  sample_count = {sample_count}  (unchanged)")

# Why? Python decided 'sample_count' is LOCAL because it appears on left side
# of an assignment. But it tries to read it BEFORE assigning. Hence the error.

# The right design - don't rely on global state, use parameters + return
def add_samples_correct(current_count, n):
    return current_count + n

sample_count = add_samples_correct(sample_count, 5)
print(f"\nWith correct design: {sample_count}")

Before: sample_count = 0
  UnboundLocalError: cannot access local variable 'sample_count' where it is not associated with a value
After:  sample_count = 0  (unchanged)

With correct design: 5


### Practical 3 - sorted() with multiple criteria (stable sort)
**Context:** You need to sort patients by risk score (descending), and within the same risk score, by name (alphabetical). One lambda handles both.

In [21]:
patients = [
    {"name": "Bilal",  "risk": 8, "age": 45},
    {"name": "Ahmad",  "risk": 8, "age": 34},
    {"name": "Sara",   "risk": 5, "age": 28},
    {"name": "Zara",   "risk": 8, "age": 51},
    {"name": "Hassan", "risk": 3, "age": 67},
    {"name": "Nida",   "risk": 5, "age": 39},
]

# Single-criterion sort
by_risk = sorted(patients, key=lambda p: p["risk"], reverse=True)
print("By risk only (ties arbitrary):")
for p in by_risk:
    print(f"  Risk {p['risk']}  {p['name']}")

# Multi-criterion sort - tuple key
# Python sorts tuples element by element (left to right)
by_risk_then_name = sorted(patients,
                           key=lambda p: (-p["risk"], p["name"]))
print("\nBy risk desc, then name asc (ties resolved):")
for p in by_risk_then_name:
    print(f"  Risk {p['risk']}  {p['name']}")

By risk only (ties arbitrary):
  Risk 8  Bilal
  Risk 8  Ahmad
  Risk 8  Zara
  Risk 5  Sara
  Risk 5  Nida
  Risk 3  Hassan

By risk desc, then name asc (ties resolved):
  Risk 8  Ahmad
  Risk 8  Bilal
  Risk 8  Zara
  Risk 5  Nida
  Risk 5  Sara
  Risk 3  Hassan


### Practical 4 - import styles and namespace pollution
**Context:** Two students import the same functions differently. One of them gets a mysterious bug three cells later.

In [22]:
# Style 1: import the module (SAFEST)
import math
import random

result1 = math.sqrt(16)
print(f"math.sqrt(16) = {result1}")

# Style 2: from module import specific names (FINE for known names)
from math import pi, e, log
print(f"pi = {pi:.4f},  e = {e:.4f}")

# Style 3: from module import * (DANGEROUS - pollutes namespace)
# from math import *
# This pulls in: sqrt, sin, cos, tan, log, pi, e, inf, nan, ...
# And it SILENTLY overwrites any variable you already have named 'log', 'pi', etc.

# Live demonstration of the hazard:
log = "activity log"           # a perfectly reasonable variable name
from math import log           # SILENTLY overwrites it!
print(f"\nlog is now: {log}")   # it's the math.log function, not your string!

# The safe rule:
# - Use 'import module' for full modules
# - Use 'from module import name' only for specific things you need
# - Never use 'from module import *' in production code

math.sqrt(16) = 4.0
pi = 3.1416,  e = 2.7183

log is now: <built-in function log>


## 9. Practical Exercise - Reusable Analytics Toolkit (**`Optional`**)
**Difficulty: Medium–High**

Build a module-style analytics library. Each function does one thing well, handles edge cases, and can be composed with others.

In [24]:
import math
from collections import Counter

def descriptive_summary(data):
    """
    Return a summary dict: mean, median, mode, std, iq_range, skewness_direction.
    No numpy or statistics module - everything from scratch.
    """
    if not data:
        return None

    n    = len(data)
    mean = sum(data) / n

    # Median
    srt    = sorted(data)
    median = srt[n//2] if n % 2 else (srt[n//2-1] + srt[n//2]) / 2

    # Mode - most frequent value
    counts = Counter(data)
    mode   = counts.most_common(1)[0][0]

    # Standard deviation (population)
    variance = sum((x - mean) ** 2 for x in data) / n
    std      = math.sqrt(variance)

    # IQR
    q1 = srt[n // 4]
    q3 = srt[3 * n // 4]

    # Skewness direction
    if mean > median * 1.05:
        skew = "right-skewed"
    elif mean < median * 0.95:
        skew = "left-skewed"
    else:
        skew = "approximately symmetric"

    return {
        "n": n, "mean": round(mean, 4), "median": median,
        "mode": mode, "std": round(std, 4),
        "iq_range": q3 - q1, "q1": q1, "q3": q3,
        "skewness": skew
    }

# Test
income = [25000, 28000, 32000, 35000, 29000, 31000, 27000, 150000]
summary = descriptive_summary(income)
print("Descriptive summary for income data:")
for k, v in summary.items():
    print(f"  {k:<18} {v}")

Descriptive summary for income data:
  n                  8
  mean               44625.0
  median             30000.0
  mode               25000
  std                39934.1254
  iq_range           7000
  q1                 28000
  q3                 35000
  skewness           right-skewed


In [24]:
def normalize(data, method="minmax"):
    """
    Normalize a list of numbers.
    method='minmax': scale to [0, 1]
    method='zscore': subtract mean, divide by std
    """
    if not data or len(set(data)) == 1:
        return [0.0] * len(data)   # all same value - degenerate case

    if method == "minmax":
        lo, hi = min(data), max(data)
        return [round((x - lo) / (hi - lo), 4) for x in data]

    elif method == "zscore":
        mean = sum(data) / len(data)
        std  = math.sqrt(sum((x - mean)**2 for x in data) / len(data))
        return [round((x - mean) / std, 4) for x in data]

    else:
        raise ValueError(f"Unknown method: {method!r}. Use 'minmax' or 'zscore'")

# Test both methods
data = [25000, 28000, 32000, 35000, 29000, 31000, 27000]
mm  = normalize(data, "minmax")
zscore = normalize(data, "zscore")

print(f"Original: {data}")
print(f"Min-max:  {mm}")
print(f"Z-score:  {zscore}")
print(f"\nMin-max range:  [{min(mm):.2f}, {max(mm):.2f}]")
print(f"Z-score mean:   {sum(zscore)/len(zscore):.4f}  (should be 0)")

Original: [25000, 28000, 32000, 35000, 29000, 31000, 27000]
Min-max:  [0.0, 0.3, 0.7, 1.0, 0.4, 0.6, 0.2]
Z-score:  [-1.4698, -0.5052, 0.7808, 1.7454, -0.1837, 0.4593, -0.8268]

Min-max range:  [0.00, 1.00]
Z-score mean:   0.0000  (should be 0)


In [25]:
def outlier_flags(data, method="iqr"):
    """
    Return a list of bools: True where a value is an outlier.
    method='iqr':    1.5 * IQR fence rule
    method='zscore': |z| > 3
    """
    n   = len(data)
    srt = sorted(data)

    if method == "iqr":
        q1     = srt[n // 4]
        q3     = srt[3 * n // 4]
        iqr    = q3 - q1
        lo     = q1 - 1.5 * iqr
        hi     = q3 + 1.5 * iqr
        return [x < lo or x > hi for x in data]

    elif method == "zscore":
        mean = sum(data) / n
        std  = math.sqrt(sum((x - mean)**2 for x in data) / n)
        if std == 0:
            return [False] * n
        return [abs((x - mean) / std) > 3 for x in data]

    else:
        raise ValueError(f"Unknown method: {method!r}")

# Test
salaries = [45000, 47000, 43000, 46000, 44000, 48000, 200000, 42000]
iqr_flags    = outlier_flags(salaries, "iqr")
zscore_flags = outlier_flags(salaries, "zscore")

print(f"Salaries: {salaries}")
print(f"IQR flags:    {iqr_flags}")
print(f"Z-score flags:{zscore_flags}")

# Separate outliers from clean data
clean    = [x for x, flag in zip(salaries, iqr_flags) if not flag]
outliers = [x for x, flag in zip(salaries, iqr_flags) if flag]
print(f"\nClean:    {clean}  mean={sum(clean)/len(clean):.0f}")
print(f"Outliers: {outliers}")

Salaries: [45000, 47000, 43000, 46000, 44000, 48000, 200000, 42000]
IQR flags:    [False, False, False, False, False, False, True, False]
Z-score flags:[False, False, False, False, False, False, False, False]

Clean:    [45000, 47000, 43000, 46000, 44000, 48000, 42000]  mean=45000
Outliers: [200000]


In [26]:
def correlate(x, y):
    """
    Pearson correlation coefficient between two lists.
    Returns float in [-1, +1].
    """
    if len(x) != len(y):
        raise ValueError(f"Lists must be same length: {len(x)} vs {len(y)}")
    if len(x) < 2:
        raise ValueError("Need at least 2 data points")

    n    = len(x)
    mx   = sum(x) / n
    my   = sum(y) / n
    num  = sum((xi - mx) * (yi - my) for xi, yi in zip(x, y))
    den  = math.sqrt(sum((xi - mx)**2 for xi in x) *
                     sum((yi - my)**2 for yi in y))

    return round(num / den, 4) if den != 0 else 0.0

# Test correlations
study_hours = [2, 3, 5, 4, 6, 7, 8, 5, 3, 9]
exam_scores = [55, 60, 75, 70, 80, 85, 92, 72, 58, 95]
sleep_hours = [8, 7, 6, 7, 5, 5, 4, 6, 7, 3]

print(f"Study hours vs Exam scores: r = {correlate(study_hours, exam_scores)}")
print(f"Sleep hours vs Exam scores: r = {correlate(sleep_hours, exam_scores)}")
print(f"Study hours vs Sleep hours: r = {correlate(study_hours, sleep_hours)}")

def interpret_r(r):
    a = abs(r)
    direction = "positive" if r > 0 else "negative"
    if a >= 0.7:   strength = "strong"
    elif a >= 0.4: strength = "moderate"
    else:          strength = "weak"
    return f"{strength} {direction} correlation"

for pair, r in [("Study-Score", correlate(study_hours, exam_scores)),
                ("Sleep-Score", correlate(sleep_hours, exam_scores))]:
    print(f"  {pair}: {r:.4f} - {interpret_r(r)}")

Study hours vs Exam scores: r = 0.9914
Sleep hours vs Exam scores: r = -0.9634
Study hours vs Sleep hours: r = -0.9855
  Study-Score: 0.9914 - strong positive correlation
  Sleep-Score: -0.9634 - strong negative correlation


In [27]:
# Full CLI report - pulls everything together
def analytics_report(data, label="Dataset"):
    """Print a formatted analytics report for a numeric dataset."""
    print(f"\n{'='*50}")
    print(f"Analytics Report: {label}")
    print(f"{'='*50}")

    s = descriptive_summary(data)
    print(f"  N             : {s['n']}")
    print(f"  Mean          : {s['mean']:,.2f}")
    print(f"  Median        : {s['median']:,.2f}")
    print(f"  Mode          : {s['mode']:,.2f}")
    print(f"  Std Dev       : {s['std']:,.2f}")
    print(f"  IQR           : {s['iq_range']:,.2f}")
    print(f"  Distribution  : {s['skewness']}")

    flags = outlier_flags(data)
    outlier_vals = [x for x, f in zip(data, flags) if f]
    print(f"  Outliers (IQR): {len(outlier_vals)} - {outlier_vals}")

    normed = normalize(data)
    print(f"  Normalised min: {min(normed):.4f}")
    print(f"  Normalised max: {max(normed):.4f}")

    # Text histogram
    srt = sorted(data)
    lo, hi = srt[0], srt[-1]
    bucket_size = (hi - lo) / 5 if hi > lo else 1
    print("\n  Distribution (5 buckets):")
    for i in range(5):
        b_lo = lo + i * bucket_size
        b_hi = b_lo + bucket_size
        count = sum(1 for x in data if b_lo <= x < b_hi)
        bar   = "█" * count
        print(f"    {b_lo:>8.1f} – {b_hi:<8.1f} {bar} ({count})")

# Run it
exam_scores = [88, 55, 94, 43, 77, 91, 60, 82, 38, 96, 72, 68, 85, 58, 79]
analytics_report(exam_scores, "Exam Scores")

salaries = [45000, 47000, 43000, 46000, 44000, 48000, 200000, 42000, 44500]
analytics_report(salaries, "Salary Data")


Analytics Report: Exam Scores
  N             : 15
  Mean          : 72.40
  Median        : 77.00
  Mode          : 88.00
  Std Dev       : 17.62
  IQR           : 30.00
  Distribution  : left-skewed
  Outliers (IQR): 0 - []
  Normalised min: 0.0000
  Normalised max: 1.0000

  Distribution (5 buckets):
        38.0 – 49.6     ██ (2)
        49.6 – 61.2     ███ (3)
        61.2 – 72.8     ██ (2)
        72.8 – 84.4     ███ (3)
        84.4 – 96.0     ████ (4)

Analytics Report: Salary Data
  N             : 9
  Mean          : 62,166.67
  Median        : 45,000.00
  Mode          : 45,000.00
  Std Dev       : 48,763.60
  IQR           : 3,000.00
  Distribution  : right-skewed
  Outliers (IQR): 1 - [200000]
  Normalised min: 0.0000
  Normalised max: 1.0000

  Distribution (5 buckets):
     42000.0 – 73600.0  ████████ (8)
     73600.0 – 105200.0  (0)
    105200.0 – 136800.0  (0)
    136800.0 – 168400.0  (0)
    168400.0 – 200000.0  (0)


## Summary

| Concept | Key point |
|---|---|
| `def` + `return` | Functions are reusable, named units of logic - if you repeat code, extract a function |
| Mutable default | `def f(x, y=[])` - the list is created ONCE. Use `None` as default, create inside |
| `*args` | Collects extra positional args into a **tuple** |
| `**kwargs` | Collects extra keyword args into a **dict** |
| LEGB scope | Python looks up names: Local → Enclosing → Global → Built-in |
| `lambda` | Single-expression anonymous function - best used as `key=` in `sorted()`, `map()`, `filter()` |
| `map(f, it)` / `filter(f, it)` | Return iterators - wrap with `list()`. Equivalent to comprehensions but more composable |
| `Counter` | Fastest way to count frequencies - `Counter(list).most_common(n)` |
| `defaultdict` | Eliminates `KeyError` for missing keys - auto-initialises with a factory |

**Module 1 complete.** You now have the full Python foundation: types, strings, collections (lists, tuples, dicts, sets), control flow, and functions. Everything from here - NumPy, Pandas, Matplotlib, statistics - builds on exactly this.

Revise using the Module 1 question bank before the assessment.

# **Addiotional Concepts (You can safely Ignore Optional Sections)**

## 8. Positional-Only (`/`) and Keyword-Only (`*`) Arguments (**`Optional`**)

Two parameter modifiers introduced in Python 3.8 that give you precise
control over **how** callers must pass arguments.

- Parameters **before `/`** - positional-only: must be given by position, cannot use keyword syntax
- Parameters **after `*`** - keyword-only: must be given by name, cannot be positional

Both solve real problems in API design and prevent a whole class of bugs
in data pipelines where argument order is easy to mix up.


In [ ]:
import math

# Positional-only: parameters before /
# Used when: the parameter name is an implementation detail, not part of the API,
# OR when positional speed matters (e.g. C extension wrappers)

def circle_area(r, /):
    """Radius must be passed positionally. Name 'r' is not part of API."""
    return math.pi * r ** 2

print(circle_area(5))          # OK - positional
# print(circle_area(r=5))      # TypeError - r is positional-only
                                # Uncomment to see the error

78.53981633974483


In [ ]:
# Keyword-only: parameters after *
# Used when: we have boolean flags or options that are confusing without names
# This forces callers to write normalize([1,2,3], method="zscore")
# NOT   normalize([1,2,3], "zscore")  - which looks like a data argument

def normalize(data, *, method="minmax"):
    """method must be passed as a keyword argument - prevents silent mistakes."""
    if method == "minmax":
        lo, hi = min(data), max(data)
        return [round((x - lo) / (hi - lo), 4) for x in data]
    elif method == "zscore":
        mean = sum(data) / len(data)
        std  = math.sqrt(sum((x - mean)**2 for x in data) / len(data))
        return [round((x - mean) / std, 4) for x in data] if std else [0.0]*len(data)
    else:
        raise ValueError(f"Unknown method: {method!r}. Choose 'minmax' or 'zscore'.")

print(normalize([10, 20, 30, 40], method="minmax"))
print(normalize([10, 20, 30, 40], method="zscore"))
# print(normalize([10,20,30], "minmax"))   # TypeError - method is keyword-only

78.53981633974483
[0.0, 0.3333, 0.6667, 1.0]
[-1.3416, -0.4472, 0.4472, 1.3416]


### 8a. Combining / and * in one function

In [5]:
# following is the full signature form:
# def f(pos_only, /, normal, *, kw_only)

def run_pipeline(data, /, label="result", *, verbose=False, precision=4):
    """
    data       - positional-only   (implementation detail, don't expose name)
    label      - normal            (can be positional OR keyword)
    verbose    - keyword-only      (flag, must be named to be clear)
    precision  - keyword-only      (option, must be named to be clear)
    """
    mean = sum(data) / len(data)
    result = round(mean, precision)
    if verbose:
        print(f"[{label}] n={len(data)}  mean={result}")
    return result

# All valid calls:
run_pipeline([88, 74, 91], verbose=True)
run_pipeline([88, 74, 91], "cohort_1", verbose=True, precision=2)
run_pipeline([88, 74, 91], label="test", verbose=False)

# Invalid - uncomment to verify:
# run_pipeline(data=[1,2,3])         # TypeError: data is positional-only
# run_pipeline([1,2,3], True)        # TypeError: verbose is keyword-only

[result] n=3  mean=84.3333
[cohort_1] n=3  mean=84.33


84.3333

## 9. Type Hinting and Annotations (**`Recommended` for Production Grade Pipelines**)

Type hints are **optional metadata** that tell readers (and IDEs and static
analysis tools like `mypy`) what types a function expects and returns.
```python
Type hinting is like adding a labeling system to your code. It lets you explicitly state, "Hey, this function expects a string, and it promises to return a float."
```
Python does **not enforce** them at runtime - they are documentation, not
constraints. But they change the experience of working with code dramatically:
your IDE auto-completes correctly, mismatched types are flagged before you run,
and the function signature becomes self-explanatory.
Imagine you are working on a massive data science pipeline and you come across a function written by a coworker, Without reading the entire function, you have no idea what to pass in. Is records a list, a dictionary, or a database connection? Is scale_factor an integer or a boolean?

```python

def process_data(records: list[dict[str, Any]], scale_factor: float, debug: bool) -> list[float]:
    # ...
```

Just by looking at the signature (the first line), you instantly know:
- `records` must be a list of dictionaries where the keys are strings.
- `scale_factor` must be a decimal number (float).
-  `debug` must be a True/False flag (boolean).
- The function is going to `return a brand new list of decimal numbers`.

In [ ]:
#Basic annotations
# Syntax: parameter: type   and   -> return_type

def gc_content(sequence: str) -> float:
    """Return GC content as a percentage."""
    seq = sequence.upper()
    gc  = seq.count("G") + seq.count("C")
    return gc / len(seq) * 100 if seq else 0.0

def mean(values: list) -> float:
    """Return the arithmetic mean of a list of numbers."""
    return sum(values) / len(values) if values else 0.0

print(f"GC content: {gc_content('ATCGATCG'):.1f}%")
print(f"Mean:       {mean([88, 74, 91, 65, 78]):.2f}")

# Annotations are stored in __annotations__ - Python never checks them
print(f"\ngc_content annotations: {gc_content.__annotations__}")

GC content: 50.0%
Mean:       79.20

gc_content annotations: {'sequence': <class 'str'>, 'return': <class 'float'>}


### 9a. Complex type hints

1. Classes vs. Functions (Types vs. Creating Objects) - list and str (Without Parentheses): These represent the Type (or Class) itself. They are the blueprint.
2. list() and str() (With Parentheses): These are constructor calls. When you add parentheses, you are telling Python to actively run a function to create a brand new, empty object in memory right now.

```python
x = list   # x is now pointing to the "list" type itself
y = list() # y is a brand-new, physical empty list object: []
```

In [6]:
# Python 3.10+: use built-in generics directly
#   list[float]    dict[str, int]    tuple[str, int]
# Python 3.9:     same
# Python 3.8:     from typing import List, Dict, Tuple, Optional

from typing import Optional, Union

# Optional[X] means the value can be X or None
def parse_age(raw: str) -> Optional[int]:
    """Extract integer age from a messy string. Returns None if unparseable."""
    import re
    found = re.findall(r'\d+', raw)
    return int(found[0]) if found else None

# Union[X, Y] means the value can be X or Y
def format_value(v: Union[int, float], unit: str = "") -> str:
    return f"{v:.2f} {unit}".strip()

# dict return type
def descriptive(data: list[float]) -> dict[str, float]:
    n = len(data)
    m = sum(data) / n
    return {"mean": round(m, 4), "n": n, "min": min(data), "max": max(data)}

print(parse_age("28 years old"))   # 28
print(parse_age("N/A"))            # None
print(format_value(38.9, "°C"))
print(descriptive([88, 74, 91, 65, 78]))

28
None
38.90 °C
{'mean': 79.2, 'n': 5, 'min': 65, 'max': 91}


In [8]:
# *args and **kwargs with type hints

def mean_typed(*values: float) -> float:
    """Type hint on *args annotates the TYPE OF EACH ELEMENT, not the tuple."""
    return sum(values) / len(values) if values else 0.0

def create_record(**fields: str) -> dict[str, str]:
    """Type hint on **kwargs annotates the TYPE OF EACH VALUE."""
    return {k: v.strip().title() for k, v in fields.items()}

print(mean_typed(88.0, 74.5, 91.0, 65.0))
print(create_record(name="  AHMAD raza  ", dept="computer science"))

# #Function as parameter
# from typing import Callable

# def apply_to_all(data: list[float],
#                  func: Callable[[float], float]) -> list[float]:
#     """Apply any single-arg float -> float function to every element."""
#     return [func(x) for x in data]

# import math
# print(apply_to_all([1, 4, 9, 16, 25], math.sqrt))

#Quick reminder: Python does NOT enforce these
# print(mean_typed("this", "is", "wrong"))   # still runs - no TypeError at runtime
# Use mypy for actual enforcement:  mypy script.py

79.625
{'name': 'Ahmad Raza', 'dept': 'Computer Science'}


## 10. Docstrings and Documentation Standards (**`Optional` but `Recommended`**)

A docstring is the first string literal inside a function, class, or module.
Python stores it in `__doc__` and tools like `help()`, Sphinx, and Jupyter's
`?` operator display it automatically.

Three widely used formats in data science:

| Style | Used by |
|---|---|
| **Google** | Google, fast to write, easy to read |
| **NumPy/SciPy** | NumPy, Pandas, SciPy, most scientific Python |
| **reStructuredText** | Sphinx default, older codebases |

For this course we use the **NumPy style** - it's what you'll find in every
data science library you use.


In [11]:
#One-liner docstring (for obvious functions)
def square(x: float) -> float:
    """Return the square of x."""
    return x ** 2

# Google style 
def gc_content_google(sequence: str) -> float:
    """Calculate the GC content of a DNA sequence.

    Args:
        sequence (str): DNA sequence. Upper or lowercase accepted.

    Returns:
        float: GC percentage in range [0.0, 100.0].

    Raises:
        ValueError: If sequence is empty.

    Examples:
        >>> gc_content_google('ATCGATCG')
        50.0
    """
    if not sequence:
        raise ValueError("Sequence cannot be empty")
    seq = sequence.upper()
    return (seq.count('G') + seq.count('C')) / len(seq) * 100

# NumPy / SciPy style (recommended for data science)
def descriptive_summary(data: list[float],
                        ddof: int = 1) -> dict[str, float]:
    """Compute descriptive statistics for a numeric list.

    Parameters
    ----------
    data : list of float
        Input data. Must contain at least 2 elements.
    ddof : int, optional
        Delta degrees of freedom for std computation. Default is 1 (sample std).

    Returns
    -------
    dict
        Dictionary with keys: n, mean, median, std, min, max.

    Raises
    ------
    ValueError
        If data has fewer than 2 elements.

    Examples
    --------
    >>> descriptive_summary([88, 74, 91, 65, 78])
    {'n': 5, 'mean': 79.2, 'median': 78, 'std': 10.26, 'min': 65, 'max': 91}
    """
    if len(data) < 2:
        raise ValueError(f"Need at least 2 elements, got {len(data)}")
    n    = len(data)
    mean = sum(data) / n
    srt  = sorted(data)
    med  = srt[n//2] if n % 2 else (srt[n//2-1] + srt[n//2]) / 2
    std  = (sum((x - mean)**2 for x in data) / (n - ddof)) ** 0.5
    return {"n": n, "mean": round(mean, 2), "median": med,
            "std": round(std, 2), "min": min(data), "max": max(data)}

# Access docstrings
help(descriptive_summary)     # prints formatted docstring
print(gc_content_google.__doc__[:80])  # raw string

Help on function descriptive_summary in module __main__:

descriptive_summary(data: list[float], ddof: int = 1) -> dict[str, float]
    Compute descriptive statistics for a numeric list.

    Parameters
    ----------
    data : list of float
        Input data. Must contain at least 2 elements.
    ddof : int, optional
        Delta degrees of freedom for std computation. Default is 1 (sample std).

    Returns
    -------
    dict
        Dictionary with keys: n, mean, median, std, min, max.

    Raises
    ------
    ValueError
        If data has fewer than 2 elements.

    Examples
    --------
    >>> descriptive_summary([88, 74, 91, 65, 78])
    {'n': 5, 'mean': 79.2, 'median': 78, 'std': 10.26, 'min': 65, 'max': 91}

Calculate the GC content of a DNA sequence.

Args:
    sequence (str): DNA seque


In [13]:
# #Doctest: runnable examples inside docstrings
# # The >>> lines in docstrings can be executed as tests with doctest.testmod()

# def fahrenheit_to_celsius(f: float) -> float:
#     """Convert Fahrenheit to Celsius.

#     Parameters
#     ----------
#     f : float
#         Temperature in Fahrenheit.

#     Returns
#     -------
#     float
#         Temperature in Celsius, rounded to 1 decimal place.

#     Examples
#     --------
#     >>> fahrenheit_to_celsius(98.6)
#     37.0
#     >>> fahrenheit_to_celsius(32.0)
#     0.0
#     >>> fahrenheit_to_celsius(212.0)
#     100.0
#     """
#     return round((f - 32) * 5 / 9, 1)

# # Run all >>> examples as tests
# import doctest
# results = doctest.testmod(verbose=False)
# print(f"Doctest results: {results.attempted} tests, {results.failed} failures")
# print(f"fahrenheit_to_celsius(98.6) = {fahrenheit_to_celsius(98.6)}")

## 11. Generator Functions (`yield`) (**`Optional`**)

A generator function uses `yield` instead of `return`. Every time you call
`next()` on it (or loop over it), it resumes from where it left off.

**Why does this matter for data science?**

A regular function that reads a 10 GB FASTA file and returns a list would
consume 10 GB of RAM. A generator reads one record at a time - memory usage
stays constant no matter how large the file is.

Generators are the right tool whenever you process sequences of things
and don't need everything in memory at once.


In [9]:
#Simple generator - count up forever without using infinite memory

def count_up(start: int = 0):
    """Infinite counter - yields one integer at a time."""
    n = start
    while True:
        yield n
        n += 1

counter = count_up(1)
print(next(counter))   # 1
print(next(counter))   # 2
print(next(counter))   # 3
# The generator is paused here - n=4 is waiting

# for loop calls next() automatically and stops at StopIteration
def first_n(start: int, n: int):
    """Yield the first n integers starting from start."""
    for i in range(n):
        yield start + i

print(list(first_n(10, 5)))   # [10, 11, 12, 13, 14]

# Generator expression (like a list comprehension but lazy)
squares_gen = (x**2 for x in range(1, 6))
print(type(squares_gen))           # <class 'generator'>
print(list(squares_gen))           # [1, 4, 9, 16, 25]

1
2
3
[10, 11, 12, 13, 14]
<class 'generator'>
[1, 4, 9, 16, 25]


In [10]:
#  Generator vs list: memory comparison  
import sys

n = 1_000_000

list_version = [x**2 for x in range(n)]          # builds entire list in RAM
gen_version  = (x**2 for x in range(n))          # builds NOTHING yet

print(f"List of 1M squares:      {sys.getsizeof(list_version):>12,} bytes")
print(f"Generator for 1M squares:{sys.getsizeof(gen_version):>12,} bytes")
print(f"Memory saved: {sys.getsizeof(list_version) / sys.getsizeof(gen_version):.0f}x")

List of 1M squares:         8,448,728 bytes
Generator for 1M squares:         200 bytes
Memory saved: 42244x


In [11]:
# ASTA parser generator - the most common bioinformatics generator pattern  

def parse_fasta(text: str):
    """
    Parse multi-FASTA text and yield (header, sequence) tuples one at a time.
    Memory cost: one record at a time, not the whole file.

    Parameters
    ----------
    text : str
        Raw FASTA-format text (headers start with '>').

    Yields
    ------
    tuple[str, str]
        (header_without_>, sequence_as_single_string)
    """
    header     = None
    seq_lines  = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if header is not None:           # yield the PREVIOUS record
                yield header, "".join(seq_lines)
            header    = line[1:]             # strip the >
            seq_lines = []
        else:
            seq_lines.append(line.upper())
    if header is not None:                   # yield the last record
        yield header, "".join(seq_lines)


# Test with inline FASTA string (same as reading a .fasta file line by line)
fasta_text = """
>BRCA1|chr17|43044295-43125483
ATCGATCGATCGATCGGCTAGCTAGCTA
GCTAGCTAGCTAGCTAGCTAGCTAGCTA
>TP53|chr17|7668421-7687550
GCGATCGATCGATCGATCGATCGCTAGC
TAGCTAGCTAGCTAGCTAGCTAGCTAGC
>KRAS|chr12|25205246-25250929
ATGATGATGATGATGATGATGATGATGA
TGATGATGATGATGATGATGATGATGAT
"""

# Process one record at a time - scales to gigabyte FASTA files
for header, seq in parse_fasta(fasta_text):
    gc = (seq.count("G") + seq.count("C")) / len(seq) * 100
    print(f"{header.split('|')[0]:<8}  len={len(seq):>3}  GC={gc:.1f}%")

BRCA1     len= 56  GC=50.0%
TP53      len= 56  GC=53.6%
KRAS      len= 56  GC=32.1%


In [12]:
# Generator pipeline - chain generators without materialising intermediate lists

def read_sequences(text: str):
    """Yield sequences from FASTA text (skip headers and blanks)."""
    for line in text.splitlines():
        line = line.strip()
        if line and not line.startswith(">"):
            yield line.upper()

def filter_min_length(sequences, min_len: int = 20):
    """Yield only sequences at least min_len bases long."""
    for seq in sequences:
        if len(seq) >= min_len:
            yield seq

def add_gc(sequences):
    """Yield (sequence, gc_pct) tuples."""
    for seq in sequences:
        gc = (seq.count("G") + seq.count("C")) / len(seq) * 100
        yield seq, round(gc, 2)

# Chain: read -> filter -> annotate
# Nothing is computed until we iterate - pure lazy evaluation
pipeline = add_gc(filter_min_length(read_sequences(fasta_text), min_len=25))

print("Sequences >= 25 bp with GC content:")
for seq, gc in pipeline:
    print(f"  {seq[:30]}...  GC={gc}%")

# ── yield with map and filter (functional + generator)  ─────────────────────
scores = [88, 55, 94, 43, 77, 91, 60, 82, 38, 96]

# map returns a generator - only computed when consumed
normalised = map(lambda x: round((x - min(scores)) / (max(scores) - min(scores)), 2),
                 scores)
# filter returns a generator
above_avg  = filter(lambda x: x > sum(scores)/len(scores), scores)

print(f"\nNormalised: {list(normalised)}")
print(f"Above average: {list(above_avg)}")

Sequences >= 25 bp with GC content:
  ATCGATCGATCGATCGGCTAGCTAGCTA...  GC=50.0%
  GCTAGCTAGCTAGCTAGCTAGCTAGCTA...  GC=50.0%
  GCGATCGATCGATCGATCGATCGCTAGC...  GC=57.14%
  TAGCTAGCTAGCTAGCTAGCTAGCTAGC...  GC=50.0%
  ATGATGATGATGATGATGATGATGATGA...  GC=32.14%
  TGATGATGATGATGATGATGATGATGAT...  GC=32.14%

Normalised: [0.86, 0.29, 0.97, 0.09, 0.67, 0.91, 0.38, 0.76, 0.0, 1.0]
Above average: [88, 94, 77, 91, 82, 96]


## 12. Exception Handling in Data Pipelines (**`Optional`**)

Real data is dirty. Files are missing. Columns have wrong types. Network
requests time out. A pipeline that crashes on the first bad record is useless.

Python's `try / except / else / finally` block lets you handle errors
gracefully - log them, skip them, or decide they're fatal - without stopping
the whole run.

The golden rule: **be specific**. `except Exception` is almost always wrong.
Catch exactly what you know can go wrong.


In [13]:
#Anatomy of try/except/else/finally

def safe_divide(a: float, b: float) -> float | None:
    try:
        result = a / b              # the thing that might go wrong
    except ZeroDivisionError:
        print(f"  Cannot divide {a} by zero")
        return None
    else:
        # runs ONLY if no exception was raised in try
        print(f"  {a} / {b} = {result:.4f}")
        return result
    finally:
        # runs ALWAYS - even if there was an exception, even if we return
        print(f"  [audit] safe_divide({a}, {b}) called")

print("Case 1: normal")
safe_divide(10, 3)
print("\nCase 2: zero division")
safe_divide(10, 0)

Case 1: normal
  10 / 3 = 3.3333
  [audit] safe_divide(10, 3) called

Case 2: zero division
  Cannot divide 10 by zero
  [audit] safe_divide(10, 0) called


In [14]:
# Multiple except clauses - most specific first

def parse_patient_record(raw: str) -> dict:
    """Parse a pipe-delimited patient record string.
    
    Parameters
    ----------
    raw : str
        Format: 'patient_id|name|age|temperature'
    
    Returns
    -------
    dict
        Cleaned patient record, or partial record with error info.
    """
    try:
        parts = raw.split("|")
        return {
            "id":    parts[0].strip(),
            "name":  parts[1].strip().title(),
            "age":   int(parts[2].strip()),
            "temp":  float(parts[3].strip()),
        }
    except IndexError:
        return {"error": "record has too few fields", "raw": raw}
    except ValueError as e:
        return {"error": f"type conversion failed: {e}", "raw": raw}
    except Exception as e:
        # catch-all as last resort - log and re-raise
        print(f"Unexpected error parsing record: {e}")
        raise

# Test with various inputs
test_records = [
    "P001|ahmad raza|34|38.9",       # valid
    "P002|sara khan|N/A|36.7",       # ValueError on age
    "P003|bilal ahmed|51",            # IndexError - missing temp
    "P004|zara malik|45|38.2",       # valid
    "P005|hassan ali|67|not_a_float", # ValueError on temp
]

results = [parse_patient_record(r) for r in test_records]
for rec in results:
    print(rec)

{'id': 'P001', 'name': 'Ahmad Raza', 'age': 34, 'temp': 38.9}
{'error': "type conversion failed: invalid literal for int() with base 10: 'N/A'", 'raw': 'P002|sara khan|N/A|36.7'}
{'error': 'record has too few fields', 'raw': 'P003|bilal ahmed|51'}
{'id': 'P004', 'name': 'Zara Malik', 'age': 45, 'temp': 38.2}
{'error': "type conversion failed: could not convert string to float: 'not_a_float'", 'raw': 'P005|hassan ali|67|not_a_float'}


In [15]:
# ── Pipeline with error accumulation - skip bad records, log all errors  ────

from typing import Optional

def run_etl_pipeline(raw_records: list[str],
                     *,
                     stop_on_error: bool = False) -> dict:
    """
    Extract, transform, and load patient records.

    Parameters
    ----------
    raw_records : list of str
        Raw pipe-delimited record strings.
    stop_on_error : bool, keyword-only
        If True, raise on first error. If False, skip and log.

    Returns
    -------
    dict
        {'clean': [...], 'errors': [...], 'summary': {...}}
    """
    clean  = []
    errors = []

    for i, raw in enumerate(raw_records, 1):
        try:
            rec = parse_patient_record(raw)
            if "error" in rec:
                raise ValueError(rec["error"])
            # Additional validation
            if not (0 < rec["age"] < 120):
                raise ValueError(f"Implausible age: {rec['age']}")
            if not (35.0 <= rec["temp"] <= 42.0):
                raise ValueError(f"Temperature out of clinical range: {rec['temp']}")
            clean.append(rec)

        except ValueError as e:
            error_entry = {"row": i, "raw": raw, "reason": str(e)}
            errors.append(error_entry)
            if stop_on_error:
                raise RuntimeError(f"Pipeline aborted at row {i}: {e}") from e

    return {
        "clean":   clean,
        "errors":  errors,
        "summary": {
            "total":   len(raw_records),
            "success": len(clean),
            "failed":  len(errors),
            "yield_pct": round(len(clean) / len(raw_records) * 100, 1)
        }
    }

# Run the pipeline on our test data
results = run_etl_pipeline(test_records, stop_on_error=False)

print(f"Pipeline Summary:")
for k, v in results["summary"].items():
    print(f"  {k:<12}: {v}")
print(f"\nClean records:")
for r in results["clean"]:
    print(f"  {r}")
print(f"\nError log:")
for e in results["errors"]:
    print(f"  Row {e['row']}: {e['reason']}  <- {e['raw']}")

Pipeline Summary:
  total       : 5
  success     : 2
  failed      : 3
  yield_pct   : 40.0

Clean records:
  {'id': 'P001', 'name': 'Ahmad Raza', 'age': 34, 'temp': 38.9}
  {'id': 'P004', 'name': 'Zara Malik', 'age': 45, 'temp': 38.2}

Error log:
  Row 2: type conversion failed: invalid literal for int() with base 10: 'N/A'  <- P002|sara khan|N/A|36.7
  Row 3: record has too few fields  <- P003|bilal ahmed|51
  Row 5: type conversion failed: could not convert string to float: 'not_a_float'  <- P005|hassan ali|67|not_a_float


In [ ]:
# Custom exception class - for domain-specific errors 
# When to create your own: when built-in exceptions are too generic and
# callers need to distinguish your specific error from other ValueErrors

class SequenceValidationError(ValueError):
    """Raised when a biological sequence fails validation."""
    def __init__(self, sequence: str, reason: str):
        self.sequence = sequence
        self.reason   = reason
        super().__init__(f"Invalid sequence {sequence!r}: {reason}")

def validate_dna(seq: str) -> str:
    """Validate and return uppercase DNA sequence."""
    seq = seq.strip().upper()
    if not seq:
        raise SequenceValidationError(seq, "empty sequence")
    invalid = set(seq) - set("ATCG")
    if invalid:
        raise SequenceValidationError(seq, f"contains non-ATCG characters: {invalid}")
    return seq

# Clean usage
test_seqs = ["ATCGATCG", "atcgatcg", "ATUG", "", "ATCG123", "GCGCGCGC"]
for s in test_seqs:
    try:
        clean = validate_dna(s)
        print(f"  OK:    {s!r:<15} -> {clean}")
    except SequenceValidationError as e:
        print(f"  ERROR: {s!r:<15} -> {e.reason}")

## 13. File Handling for Bioinformatics (**`Optional`**)

The `with` statement manages file handles automatically - it closes the file
even if an exception is raised inside the block. Always use `with open(...)`.

Bioinformatics works with a small set of text formats: FASTA, FASTQ, VCF,
BED, GFF, and plain CSV/TSV. All of them are plain text, all parsed line by
line. The patterns below cover 90% of real file work.


In [ ]:
import os
import csv

# Writing a file - the foundation 
output_path = "/tmp/patients.txt"

patients = [
    {"id": "P001", "name": "Ahmad Raza",  "age": 34, "temp": 38.9, "bp": "145/92"},
    {"id": "P002", "name": "Sara Khan",   "age": 28, "temp": 36.7, "bp": "118/76"},
    {"id": "P003", "name": "Bilal Ahmed", "age": 51, "temp": 37.1, "bp": "138/88"},
    {"id": "P004", "name": "Zara Malik",  "age": 45, "temp": 38.2, "bp": "132/84"},
    {"id": "P005", "name": "Hassan Ali",  "age": 67, "temp": 37.8, "bp": "155/96"},
]

# Write pipe-delimited file with header
with open(output_path, "w") as fh:
    fh.write("patient_id|name|age|temperature|blood_pressure\n")
    for p in patients:
        fh.write(f"{p['id']}|{p['name']}|{p['age']}|{p['temp']}|{p['bp']}\n")

print(f"Written {len(patients)} records to {output_path}")
print(f"File size: {os.path.getsize(output_path)} bytes")

In [ ]:
# reading a file line by line
print("=== Reading patients.txt ===")

with open(output_path, "r") as fh:
    header = fh.readline().strip().split("|")
    print(f"Columns: {header}")
    print()
    for line in fh:                        # iterate line by line - memory efficient
        line = line.strip()
        if not line:
            continue                        # skip blank lines
        fields = line.split("|")
        # zip() pairs column names with values
        record = dict(zip(header, fields))
        print(f"  {record['patient_id']}: {record['name']:<15} "
              f"temp={record['temperature']}°C  BP={record['blood_pressure']}")

In [ ]:
# FASTA file: write and read
fasta_path = "/tmp/sequences.fasta"

sequences = [
    ("BRCA1|chr17|tumour_suppressor",
     "ATCGATCGATCGATCGGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTA"),
    ("TP53|chr17|tumour_suppressor",
     "GCGATCGATCGATCGATCGATCGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAG"),
    ("KRAS|chr12|oncogene",
     "ATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATG"),
    ("EGFR|chr7|receptor_kinase",
     "GCTAGCTAGCTAGCTAGCTAGCTAGCTAGCATCGATCGATCGATCGATCGATCGATCGAT"),
]

# Write FASTA
with open(fasta_path, "w") as fh:
    for header, seq in sequences:
        fh.write(f">{header}\n")
        # Wrap sequence at 60 characters (standard FASTA line width)
        for i in range(0, len(seq), 60):
            fh.write(seq[i:i+60] + "\n")

print(f"Written {len(sequences)} sequences to {fasta_path}")

# Read FASTA using our generator from Section 11
def parse_fasta_file(filepath: str):
    """Yield (header, sequence) tuples from a FASTA file."""
    header    = None
    seq_lines = []
    with open(filepath, "r") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_lines)
                header    = line[1:]
                seq_lines = []
            else:
                seq_lines.append(line.upper())
        if header is not None:
            yield header, "".join(seq_lines)

print("\nParsed sequences:")
for header, seq in parse_fasta_file(fasta_path):
    gene = header.split("|")[0]
    gc   = (seq.count("G") + seq.count("C")) / len(seq) * 100
    print(f"  {gene:<8}  len={len(seq):>3} bp  GC={gc:.1f}%")

In [ ]:
# CSV with the csv module - handles quoting and commas in fields 
import csv

csv_path = "/tmp/lab_results.csv"

lab_data = [
    ["patient_id", "test",       "value", "unit",  "reference_range", "status"],
    ["P001",       "WBC",        "12.1",  "x10⁹/L", "4.0-11.0",      "HIGH"],
    ["P001",       "Hb",         "11.2",  "g/dL",   "12.0-16.0",     "LOW"],
    ["P001",       "Glucose",    "9.4",   "mmol/L", "3.9-6.1",       "HIGH"],
    ["P002",       "WBC",        "6.8",   "x10⁹/L", "4.0-11.0",     "NORMAL"],
    ["P002",       "Creatinine", "0.9",   "mg/dL",  "0.7-1.2",      "NORMAL"],
    ["P003",       "HbA1c",      "8.2",   "%",      "< 5.7",        "DIABETIC"],
]

# Write CSV
with open(csv_path, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerows(lab_data)

# Read CSV
print("Abnormal lab results:")
with open(csv_path, "r") as fh:
    reader = csv.DictReader(fh)   # first row auto-becomes field names
    for row in reader:
        if row["status"] != "NORMAL":
            print(f"  {row['patient_id']}  {row['test']:<15} "
                  f"{row['value']:>6} {row['unit']:<10}  [{row['status']}]")

In [ ]:
# VCF parser - minimal but real
# VCF (Variant Call Format) is the standard for genomic variants
# Lines starting with # are header/meta lines
# Data lines are tab-delimited: CHROM POS ID REF ALT QUAL FILTER INFO ...

vcf_path = "/tmp/variants.vcf"

vcf_content = """##fileformat=VCFv4.2
##reference=GRCh38
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
chr17\t43044295\trs80357065\tA\tT\t99\tPASS\tGENE=BRCA1;AF=0.412;IMPACT=HIGH
chr17\t7674220\trs28934578\tG\tA\t95\tPASS\tGENE=TP53;AF=0.281;IMPACT=MODERATE
chr12\t25245350\trs121913529\tC\tA\t87\tLOW_QUAL\tGENE=KRAS;AF=0.156;IMPACT=HIGH
chr7\t55191822\trs121913279\tT\tG\t99\tPASS\tGENE=EGFR;AF=0.634;IMPACT=HIGH
chr17\t43091437\trs80357346\tC\tT\t78\tPASS\tGENE=BRCA1;AF=0.089;IMPACT=LOW
"""

with open(vcf_path, "w") as fh:
    fh.write(vcf_content)

def parse_vcf(filepath: str) -> list[dict]:
    """Parse VCF file and return list of variant dicts."""
    variants = []
    with open(filepath, "r") as fh:
        for line in fh:
            line = line.strip()
            if line.startswith("##") or not line:
                continue                              # skip meta lines
            if line.startswith("#CHROM"):
                col_names = line.lstrip("#").split("\t")
                continue
            fields = line.split("\t")
            var    = dict(zip(col_names, fields))
            # Parse INFO field into sub-dict
            var["INFO"] = dict(item.split("=") for item in var["INFO"].split(";"))
            variants.append(var)
    return variants

variants = parse_vcf(vcf_path)

print(f"Loaded {len(variants)} variants\n")
high_impact = [v for v in variants if v["INFO"].get("IMPACT") == "HIGH"
               and v["FILTER"] == "PASS"]

print("HIGH impact / PASS variants:")
for v in high_impact:
    print(f"  {v['INFO']['GENE']:<8} {v['CHROM']}:{v['POS']:<12} "
          f"AF={v['INFO']['AF']}  {v['REF']}->{v['ALT']}")

## 14. Creating and Importing Your Own Package (**`Must`**)

As projects grow, you move functions out of notebooks into `.py` files.
A **module** is a single `.py` file. A **package** is a folder of modules
with an `__init__.py` file.

Professional analytics project layout:

```
my_project/
│
├── analytical_notebook.ipynb     ← your working notebook
│
└── my_analytics/                 ← your package folder
    ├── __init__.py               ← tells Python this is a package; re-exports key names
    ├── central_tendency.py       ← mean, median, mode
    └── scaling.py                ← min-max normalisation, z-scores
```

Once built, you import from your package exactly like any library:
```python
from my_analytics import mean, median
from my_analytics.scaling import zscore
import my_analytics as ma
```


In [5]:
# import styles - all four work

# Style 1: import specific names from the package (via __init__.py)
from src.my_analytics import mean, median, mode, minmax, zscore

# Style 2: import the package and use dot notation
import src.my_analytics as ma

# Style 3: import from a specific module inside the package
from src.my_analytics.scaling import minmax as mm_scale

# Style 4: import the module itself
import src.my_analytics.central_tendency as ct

# Use all four styles with the same data
data = [88, 74, 91, 65, 78, 82, 95, 70]

print("=== central_tendency ===")
print(f"mean()   via direct import : {mean(*data):.2f}")
print(f"mean()   via package alias : {ma.mean(*data):.2f}")
print(f"median() via module import : {ct.median(data)}")
print(f"mode()   via direct import : {mode(data)}")

print("\n=== scaling ===")
scaled_mm = minmax(data)
scaled_z  = zscore(data)
print(f"min-max : {scaled_mm}")
print(f"z-score : {scaled_z}")
print(f"min-max (alias): {mm_scale(data)}")

=== central_tendency ===
mean()   via direct import : 80.38
mean()   via package alias : 80.38
median() via module import : 80.0
mode()   via direct import : [65, 70, 74, 78, 82, 88, 91, 95]

=== scaling ===
min-max : [0.766667, 0.3, 0.866667, 0.0, 0.433333, 0.566667, 1.0, 0.166667]
z-score : [0.723326, -0.604748, 1.007913, -1.458509, -0.225298, 0.154151, 1.387363, -0.984197]
min-max (alias): [0.766667, 0.3, 0.866667, 0.0, 0.433333, 0.566667, 1.0, 0.166667]


In [18]:
# Verify the package with its own doctests
import doctest
import src.my_analytics.central_tendency as ct_module

results = doctest.testmod(ct_module, verbose=False)
print(f"Doctest results for central_tendency.py:")
print(f"  Tests run: {results.attempted}")
print(f"  Failures:  {results.failed}")
print(f"  Status:    {'PASS' if results.failed == 0 else 'FAIL'}")

# help() works on your package too
help(ma.zscore)

Doctest results for central_tendency.py:
  Tests run: 2
  Failures:  0
  Status:    PASS
Help on function zscore in module src.my_analytics.scaling:

zscore(data: 'list[float]', *, ddof: 'int' = 1) -> 'list[float]'
    Standardise data using z-score (subtract mean, divide by std).

    Parameters
    ----------
    data : list of float
        Input data.
    ddof : int, keyword-only
        Delta degrees of freedom. 0 = population std, 1 = sample std.

    Returns
    -------
    list of float
        Standardised values with mean ~ 0 and std ~ 1.



### Practical: extend the package - add a new module

In [ ]:
# Students: create my_analytics/dispersion.py with these functions:
#   - variance(data, ddof=1)  -> float
#   - std(data, ddof=1)       -> float
#   - iqr(data)               -> float
#   - coefficient_of_variation(data) -> float
# Then add them to __init__.py
# Then import and test here

# Starter: here is variance - you complete the rest
dispersion_starter = """
my_analytics/dispersion.py
Measures of dispersion - variance, std, IQR, CV.
"""


## Additional Summary

| Concept | Key point |
|---|---|
| `/` in signature | Parameters before `/` are positional-only - callers cannot use keyword syntax |
| `*` in signature | Parameters after `*` are keyword-only - callers must name them explicitly |
| Type hints | `def f(x: float) -> str` - documentation for IDEs and static checkers; not enforced at runtime |
| `Optional[X]` | `X or None` - use when a parameter or return value can be absent |
| `Callable[[float], float]` | Type hint for a function argument |
| `"""docstring"""` | First string in a function - stored in `__doc__`, shown by `help()` |
| NumPy docstring style | Parameters / Returns / Raises / Examples sections - standard in data science |
| `yield` | Pauses a function and emits one value; resumes on next `next()` call |
| Generator | Memory-efficient - computes one item at a time; essential for large bioinformatics files |
| `try / except / else / finally` | Structured error handling - catch specific exceptions, always-run cleanup in `finally` |
| Custom exception | Subclass `Exception` to create domain-specific error types |
| `with open(path) as fh` | Context manager - closes file automatically even if an exception is raised |
| FASTA / VCF / CSV parsing | Read line by line with a generator; use `csv.DictReader` for CSV |
| `__init__.py` | Marks a folder as a Python package; re-exports public API names |
| `from .module import name` | Relative import inside a package |
| `__version__`, `__all__` | Standard package metadata attributes |
